# 2.4 Exercise: the handed-out notebook, and a study of your own

Notebooks 2.1 and 2.2 put one block at a time on a beat and measured what it bought.
Here you drive, on the same heartbeats, and the class works on one problem together.

## What is asked of you

1. **Part 1: run this notebook as it is.** The baselines, a handed-out design with its
   spread, one search over a given space, its winner rerun, and the same winner on a
   balanced loss. You change nothing; you read what comes out and answer the questions for
   yourself.
2. **Part 2: one study of your own**, in the folder [`exercises/`](exercises/), along the
   nine steps of the [methodology page](https://raoulg.github.io/MADS-MachineLearning-course/huiswerk/your-study.html#lesson-2): one idea,
   a design that can test it, a search, and the result against the baseline. The class
   spreads over the open questions at the end of this notebook, and builds on each
   other's ideas.
3. **Hand in your runs** before the next lesson, with one command: `make share LESSON=2`.

Lesson 3 opens on the runs of the whole class: a map of which bets paid, each with its
spread, not a leaderboard. A bet that did not pay is a result on that map.

## Where your runs go

As in lesson 1: every run is written to `artefacts/optuna/studies.db` on your machine,
`make optuna-dashboard` shows it, and `make share LESSON=2` hands in the runs of lesson 2.
Part 1 writes to two studies, `lesson2-notebook` (the trials of the search) and
`lesson2-notebook-runs` (the baselines, the handed-out design, the winner, the
same winner on a balanced loss). Part 2 writes to `lesson2-study-<your study>` and its `-runs`.

**Nothing in Part 1 is seeded**, so your numbers will not be your neighbour's. The spread
between runs stands beside every difference you read.

## The problem: a heartbeat, from a patient the model has never seen

The data is notebook 2.1's: single beats from the MIT-BIH Arrhythmia Database, three
classes (N normal, S supraventricular ectopic, V ventricular ectopic), **split by
patient**. A beat of a patient the model trained on is not a new case: the next beat of
that patient looks like the last one. A model scored on beats of patients it saw learns
those patients and is scored too high (goal 1.9). So train and test hold other patients,
and so does the validation split every search here is scored on: six patients of train,
never trained on. Test is not touched in this lesson.

The metric is **macro recall**: the mean of the recalls of N, S and V. N is nine beats in
ten, so "always N" scores 89% accuracy and finds no patient; its macro recall is 1/3.
Every run logs its whole confusion matrix, so the recall of S, the accuracy and the rest
can be read next to it.

## Setup

The data, the trainer and the design come from the folder `exercises/`, the same code
your own study uses in Part 2:

| name | what it is |
|---|---|
| `HARNESS.data(config)` | the train loader (train without the six validation patients) and the validation loader |
| `HARNESS.fit(model, data, config)` | trains a model and scores it on the validation patients after every epoch |
| `Beats(Config(...), n_classes)` | the design of [`exercises/designs.py`](exercises/designs.py): kernels, blocks and options |
| `spread(make, config)` | `REPEATS` runs of one config, each from other random weights |
| `compare({name: values})` | repeated runs side by side: mean, spread, and the difference with the first in points |
| `search(space, name)` | `N_TRIALS` trials of a search space as one Optuna study |
| `RUNS` | the study `lesson2-notebook-runs`; `RUNS.log(...)` stores one run in it |

In [ ]:
import sys
import time

import datasets
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from exercises.designs import Beats, Config
from exercises.harness import CHOICES, CLASSES, HARNESS, LINEAR, TRAIN_ROWS, splits
from huggingface_hub.utils import disable_progress_bars as hub_quiet
from IPython.display import Image
from loguru import logger
from notebooktester import param
from sklearn.neighbors import KNeighborsClassifier
from torch import nn

from hpo.measure import Architecture, Comparison, Confusion, Repeats, repeat
from hpo.record import Idea, RunRecord, StudyLog
from hpo.search import Budget, Choice, Float, Int, Space

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()
hub_quiet()
optuna.logging.set_verbosity(optuna.logging.WARNING)

EPOCHS = HARNESS.epochs
REPEATS = HARNESS.repeats
N_TRIALS = param(24, test=2)
QUICK_TRIALS = param(6, test=2)  # the trials of E3 with QUICK (see after E2)
OBJECTIVE = "macro_recall"
EXAMPLE = torch.zeros(2, 187, 1)  # two beats are enough to trace the shapes

# Part 1 is the same for everyone, so its idea is the course's: "lesson 2 as handed out".
HANDED_OUT = Idea.handed_out(lesson=2)
RUNS = StudyLog.runs(
    name="lesson2-notebook-runs", idea=HANDED_OUT, objective=OBJECTIVE, choices=CHOICES
)

# the trainer's settings of every run in Part 1: the plain loss, unless a run says otherwise
TRAINER = {"dataset": "mitbih", "epochs": EPOCHS, "lr": 1e-3, "class_weight": 0.0}
# every choice of the design, as the handed-out design makes it
DESIGN = {**vars(Config()), **TRAINER}


def spread(make, config: dict) -> Repeats:
    """`REPEATS` runs of `make()`, trained with `config`, each from other random weights."""
    data = HARNESS.data(config)
    return repeat(make, lambda model: HARNESS.fit(model, data, config), REPEATS)


def compare(runs: dict[str, np.ndarray], metric: str = OBJECTIVE) -> None:
    """Repeated runs side by side. The first entry is the reference: every other one gets
    its difference with it, in points."""
    names = list(runs)
    rows = [
        Comparison.of(name, runs[name], versus=runs[names[0]] if i else None)
        for i, name in enumerate(names)
    ]
    _, ax = plt.subplots(figsize=(8, 0.5 + 0.55 * len(rows)))
    for y, row in enumerate(rows):
        ax.errorbar(row.mean, y, xerr=row.std, fmt="o", color="tab:blue", capsize=4)
        ax.scatter(row.values, [y] * len(row.values), color="tab:blue", alpha=0.3, s=14)
    ax.set_yticks(range(len(rows)), names)
    ax.invert_yaxis()
    ax.set_xlabel(metric)
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()
    display(
        pd.DataFrame(
            {
                "mean": [r.mean for r in rows],
                "std": [r.std for r in rows],
                "runs": [len(r.values) for r in rows],
                "difference (pt)": [r.points for r in rows],
            },
            index=names,
        ).style.format(
            {"mean": "{:.3f}", "std": "{:.3f}", "difference (pt)": "{:+.2f}"}, na_rep=""
        )
    )


def objective(space: Space, log: StudyLog):
    """What an Optuna study calls once per trial: draw a config, build `Beats`, train it,
    report the macro recall after every epoch, and return the last one."""

    def run_trial(trial: optuna.Trial) -> float:
        config = space.draw(trial)
        data = HARNESS.data(config)
        model = Beats(
            Config(**{k: v for k, v in config.items() if k in vars(Config())}), 3
        )
        seen = None

        def to_optuna(epoch, fit):
            nonlocal seen
            seen = fit
            trial.report(fit.scores().get(OBJECTIVE), epoch)

        try:
            seen = HARNESS.fit(model, data, config, to_optuna)
            return seen.scores().get(OBJECTIVE)
        finally:
            log.record(
                trial,
                RunRecord(
                    role="trial",
                    dataset="mitbih",
                    config=config,
                    fixed=space.fixed(),
                    train_seconds=seen.seconds if seen else 0.0,
                    curves=seen.curves if seen else None,
                    scores=seen.scores() if seen and seen.confusion else None,
                    confusion=seen.confusion if seen else None,
                    extra=dict(data.dims),
                ),
                Architecture.of(model, EXAMPLE),
            )

    return run_trial


def search(space: Space, name: str, n_trials: int = N_TRIALS):
    """`n_trials` random trials of `space` as the Optuna study `name`, and a table of its
    finished trials."""
    study = StudyLog.search(
        name=name,
        space=space,
        idea=HANDED_OUT,
        objective=OBJECTIVE,
        attrs={"origin": "notebook"},
    )
    start = time.time()
    study.optimize(objective(space, study), n_trials=n_trials)
    done = study.trials(optuna.trial.TrialState.COMPLETE)
    table = pd.DataFrame([{OBJECTIVE: t.value, **t.params} for t in done])
    print(
        f"{n_trials} trials in {time.time() - start:.0f}s; {len(done)} finished in all"
    )
    return study, table


train, valid = splits(TRAIN_ROWS)
counts = pd.DataFrame(
    {"train": train.counts(), "validation": valid.counts()}, index=CLASSES
).T
display(counts.assign(beats=counts.sum(axis=1)))

# Part 1: the notebook as handed out

## E1: the baselines, first

Three floors, in notebook 2.1's order, before any kernel: always N, the nearest
neighbour on the raw beat by cosine, and a linear model on the 187 steps. The linear one is
repeated, so you know its spread; it is the baseline every later result is read
against, here and in your own study.

In [ ]:
always_n = Confusion.of(valid.labels, torch.zeros_like(valid.labels), CLASSES)
knn = KNeighborsClassifier(n_neighbors=1, metric="cosine").fit(
    train.beats.flatten(1), train.labels
)
nearest = Confusion.of(
    valid.labels, torch.from_numpy(knn.predict(valid.beats.flatten(1))), CLASSES
)

linear_config = {**LINEAR, **TRAINER}
make_linear = lambda: nn.Sequential(nn.Flatten(), nn.Linear(187, 3))
linear_runs = spread(make_linear, linear_config)
baseline = linear_runs.values(OBJECTIVE)

RUNS.log(
    name="baseline",
    role="baseline",
    results=linear_runs,
    dataset="mitbih",
    params=HARNESS.params(linear_config, HARNESS.data(linear_config)),
    architecture=Architecture.of(make_linear(), EXAMPLE),
)
RUNS.log(
    name="1-nearest-neighbour",
    role="baseline",
    results=Repeats(runs=[nearest], seconds=0.0),
    dataset="mitbih",
)

for name, scores in [
    ("always N", always_n.scores()),
    ("1-nearest-neighbour", nearest.scores()),
    ("linear, first run", linear_runs.scores()[0]),
]:
    recall = "  ".join(f"{c} {scores.recall[c]:.2f}" for c in CLASSES)
    print(f"{name:22s} macro recall {scores.macro_recall:.3f}   recall {recall}")
compare({f"linear, {REPEATS} runs": baseline})

Read the recall per class. Always N has a recall of 1 on N and 0 on the others. Where
do the nearest neighbour and the linear model find their beats, and which class do they
both miss?

## E2: the handed-out design, and its spread

The design of [`exercises/designs.py`](exercises/designs.py) with every option off: a
kernel of 5, three blocks of 32 channels, each followed by a pool of 2. That is notebook
2.1's stack of depth 3. Open it in atlas before you run it:

```bash
atlas-code-nightly view notebooks/2_kernels_state_and_gates/exercises
```

In [ ]:
start = time.time()
make_design = lambda: Beats(Config(), n_classes=3)
design_runs = spread(make_design, DESIGN)
E2_SECONDS = time.time() - start
handed_out_design = design_runs.values(OBJECTIVE)
RUNS.log(
    name="handed-out design",
    role="hypothesis",
    results=design_runs,
    dataset="mitbih",
    params=HARNESS.params(DESIGN, HARNESS.data(DESIGN)),
    architecture=Architecture.of(make_design(), EXAMPLE),
)
compare(
    {
        f"linear, {REPEATS} runs": baseline,
        f"handed-out design, {REPEATS} runs": handed_out_design,
    }
)

## How long the rest of Part 1 takes here

E3 to E5 train about 22 times as long as E2 did, on any machine: 24 trials of a search
and a winner rerun twice. The cell below turns E2's time into minutes for the rest. A
laptop built for battery life rather than for compute can be twenty or thirty times
slower than the one this notebook was written on, and then the rest runs for hours.

`QUICK = True` keeps it short: E3 runs 6 trials instead of 24, and E4 and E5 show a
stored run instead of training one (`img/part1-*.png`, one full run of Part 1). The cell
turns `QUICK` on when the rest would take more than half an hour; set it yourself to
overrule it. E3 runs even then, because its trials measure what one trial costs on this
machine, and Part 2 turns the minutes you give your study into trials at that cost.

In [ ]:
# E3 to E5 take about this many times E2's time: as handed out, and with QUICK
REST_FULL, REST_QUICK = 22, 4
rest = E2_SECONDS * REST_FULL / 60
QUICK = rest > 30  # or set True or False yourself
print(
    f"E2 took {E2_SECONDS:.0f}s here: the rest of Part 1 takes about {rest:.0f} minutes, "
    f"about {E2_SECONDS * REST_QUICK / 60:.0f} with QUICK"
)
print(f"QUICK = {QUICK}")

## E3: the handed-out search

The search below is handed out to the whole class. Every axis but one is an idea of
lessons 1 and 2:

| axis | values | the bet it tests | goal |
|---|---|---|---|
| `kernel_size` | 3, 5, 9 | how much one kernel sees at once | [2.2](../../pages/leerdoelen/les2.html#g2-2) |
| `depth` | 1 to 5 | depth, and with it the receptive field | [2.4](../../pages/leerdoelen/les2.html#g2-4) |
| `downsample` | pool, stride | how the beat is shortened | [2.3](../../pages/leerdoelen/les2.html#g2-3) |
| `skip` | off, on | a skip around every block | [2.1](../../pages/leerdoelen/les2.html#g2-1) |
| `gate` | off, on | a gate before the head | [2.6](../../pages/leerdoelen/les2.html#g2-6) |
| `width` | 8 to 128 | capacity: more channels per block | lesson 3 |
| `lr` | 3e-4 to 3e-3 | the size of the optimizer's step | [1.7](../../pages/leerdoelen/les1.html#g1-7) |

`width` is not explained in this lesson. It is in the search because lesson 3 asks what
capacity buys, and opens on what the class's trials say about it.

As in lesson 1, the search samples at random, is not seeded and prunes nothing, so the
class's trials pooled are one random search of a few hundred trials.

In [ ]:
HANDED_OUT_SPACE = Space(
    {
        "dataset": "mitbih",
        "kernel_size": Choice([3, 5, 9]),
        "width": Int(8, 128, log=True),
        "depth": Int(1, 5),
        "downsample": Choice(["pool", "stride"]),
        "skip": Choice([False, True]),
        "gate": Choice([False, True]),
        "parallel": False,
        "fold": False,
        "excite": False,
        "state": False,
        "class_weight": 0.0,
        "lr": Float(3e-4, 3e-3, log=True),
        "epochs": EPOCHS,
    }
)
handed_out, df = search(
    HANDED_OUT_SPACE, "lesson2-notebook", n_trials=QUICK_TRIALS if QUICK else N_TRIALS
)
display(
    df.sort_values(OBJECTIVE, ascending=False)
    .head(6)
    .style.format({OBJECTIVE: "{:.3f}", "lr": "{:.1e}"})
    .hide(axis="index")
    .set_caption("the six best trials")
)

In [ ]:
axes = ["kernel_size", "width", "depth", "downsample", "skip", "gate", "lr"]
fig, panels = plt.subplots(1, len(axes), figsize=(18, 3), sharey=True)
for ax, axis in zip(panels, axes, strict=True):
    values = df[axis].astype(str) if df[axis].dtype == bool else df[axis]
    ax.scatter(values, df[OBJECTIVE], alpha=0.6)
    ax.set_xlabel(axis)
    if axis in ("width", "lr"):
        ax.set_xscale("log")
panels[0].set_ylabel(OBJECTIVE)
plt.suptitle("every trial of the search, one panel per axis", x=0.1, ha="left")
plt.tight_layout()
plt.show()
if QUICK:
    display(Image("img/part1-search.png"))

Per axis: did it move the macro recall, and in which direction? A panel without a
pattern is a finding too: within its range, that axis did not matter on 24 draws. Which
panels would you want more trials on? With `QUICK` your panels hold 6 trials; read the
stored run of 24 below them.

## E4: rerun the winner honestly

The best trial is one run, picked from 24: its score is partly luck. Rerun its config
as often as the baseline, and compare like with like. With `QUICK` the cell shows the
stored run instead: its winner is another config than yours would be, and another rerun
of the same config lands elsewhere within its spread.

In [ ]:
if QUICK:
    display(Image("img/part1-winner.png"))
else:
    best = handed_out.best()
    display(
        pd.DataFrame([{k: v for k, v in best.items() if k != "epochs"}])
        .style.hide(axis="index")
        .set_caption("the winning config")
    )
    winner_config = Config(**{k: v for k, v in best.items() if k in vars(Config())})
    make_winner = lambda: Beats(winner_config, n_classes=3)
    winner_runs = spread(make_winner, best)
    winner = winner_runs.values(OBJECTIVE)
    RUNS.log(
        name="winner",
        role="final",
        results=winner_runs,
        dataset="mitbih",
        params=HARNESS.params(best, HARNESS.data(best)),
        architecture=Architecture.of(make_winner(), EXAMPLE),
        attrs={"search_space": HANDED_OUT_SPACE.line()},
    )
    print(f"the winner's one run in the search: {df[OBJECTIVE].max():.3f}")
    compare(
        {
            f"handed-out design, {REPEATS} runs": handed_out_design,
            f"the winner rerun, {REPEATS} runs": winner,
        }
    )

1. How much of the winner's score in the search survived the rerun?
2. The table reads the winner against the handed-out design. How many points lie between
   them, beside the spread of each? What did the search buy?

## E5: the loss steers, the metric judges

Every run so far trained on the plain cross-entropy, which counts every beat once: nine
in ten are N, so the loss is mostly about N, while the metric weighs S as much as N. The
trainer's axis `class_weight` weighs each class in the loss by
`(n / (k * n_class)) ** class_weight`: 0 is the plain loss, 1 makes every class weigh as
much as the others (goal 1.8).

A loss that weighs S as much as N: what happens to the recall of S, and what does N pay
for it?

In [ ]:
if QUICK:
    display(Image("img/part1-balanced.png"))
else:
    weighted_config = {**best, "class_weight": 1.0}
    weighted_runs = spread(make_winner, weighted_config)
    RUNS.log(
        name="winner, balanced loss",
        role="control",
        results=weighted_runs,
        dataset="mitbih",
        params=HARNESS.params(weighted_config, HARNESS.data(weighted_config)),
        architecture=Architecture.of(make_winner(), EXAMPLE),
    )
    for name, runs in [("plain loss", winner_runs), ("balanced loss", weighted_runs)]:
        mean = runs.mean()
        recall = "  ".join(f"{c} {mean.recall[c]:.2f}" for c in CLASSES)
        print(
            f"{name:14s} accuracy {mean.accuracy:.3f}  macro recall {mean.macro_recall:.3f}"
            f"   recall {recall}"
        )
    compare(
        {
            f"winner, plain loss, {REPEATS} runs": winner,
            f"winner, balanced loss, {REPEATS} runs": weighted_runs.values(OBJECTIVE),
        }
    )

What did the weights trade? A loss that weighs S more finds more S and calls more N
beats S; the accuracy drops while the macro recall may rise. Which of the two numbers a
cardiologist would want is a question about the problem, not about the model.

Your runs of Part 1 are now in `lesson2-notebook` and `lesson2-notebook-runs`.

# Part 2: your own study

The loop is the one of lesson 1, on lesson 2's design: the nine steps of the
[methodology page](https://raoulg.github.io/MADS-MachineLearning-course/huiswerk/your-study.html#lesson-2), the same in every lesson.
The steps at the end of this part walk them for lesson 2. You work in
[`exercises/`](exercises/):

| file | what it is | what you do with it |
|---|---|---|
| [`exercises/designs.py`](exercises/designs.py) | `Beats`, one typed method per stage; the blocks of this lesson are options: `parallel`, `fold`, `skip`, `excite`, `state`, `gate` | read it with atlas; add a stage when your idea needs another block |
| [`exercises/study.py`](exercises/study.py) | six settings at the top (`STUDY`, `IDEA`, `EXPECTS`, `OBJECTIVE`, `SPACE`, `MINUTES`) and the code that runs the study | fill in the settings, then `make study LESSON=2` |
| [`exercises/harness.py`](exercises/harness.py) | the heartbeats split by patient, the trainer with `class_weight`, the linear baseline | nothing: it is given |

## What S is, and what your design needs to know

In notebook 2.2 every design finds about 2% of the S beats, and the blocks are not the
reason: a row does not contain what makes an S an S (section 2.2.5). An S beat is a
supraventricular ectopic beat: it starts too early above the ventricles and then travels the
usual route, so its spike looks almost like an N. What gives it away is its **timing**, too
early after the beat before it, and the P wave before it, which is different or missing. A V
beat starts in the ventricle itself and has a wide spike of another shape, which is why its
recall is about 0.7. To design for S you need this domain knowledge: what a cardiologist
looks at.

The row of the handed-out dataset starts at the R-peak, so neither the beat before nor the
P wave is in it. The full recordings are public, as a dataset of your own to use:
[`pttrn-io/ecg-mitbih-10s-raw`](https://huggingface.co/datasets/pttrn-io/ecg-mitbih-10s-raw),
every recording of MIT-BIH uncut, as windows of 10 seconds `(1250, 1)` at 125 Hz with all
the beats of each window: their R-peaks, their class (including F and Q, which the handed-out
dataset leaves out), their rhythm. Windows of one record follow each other without a gap, so
a longer stretch is a few rows glued together; the dataset's card has the loading code, the
columns and how to glue. It is split by patient like the handed-out one, and has no
validation split: carve it by record with the same `VAL_RECORDS`.

**Where to start: section 2.2.5 of notebook 2.2.** It computes, from these windows, for
every beat the interval to the beat before divided by the patient's median interval, shows
that S and V come early and N does not, and tries one rule on that number. Read that
analysis first: it is the evidence that the interval holds what the beat does not, and its
cell is code you can reuse. Two things it leaves open are yours: it takes the median over a
patient's whole recording, and it stops at a rule, not at a design.

To give a beat of the handed-out dataset its interval, join on the record and the R-peak
(`r_peaks_360` of the windows is the `r_peak` of the beats):

```python
import numpy as np
from vectormesh import VectorCache

windows = VectorCache.from_hub("pttrn-io/ecg-mitbih-10s-raw", split="train")
beats = VectorCache.from_hub("pttrn-io/heartbeats-mitbih-raw", split="train")

w = windows.dataset.with_format(None)
peaks = {}  # record -> the R-peaks of all its beats at 360 Hz, in time order
for record, row_peaks in zip(w["record"], w["r_peaks_360"]):
    peaks.setdefault(record, []).extend(row_peaks)
peaks = {record: np.asarray(p) for record, p in peaks.items()}

b = beats.dataset.with_format(None, columns=["record", "r_peak"])
before = np.full(len(b), np.nan)  # seconds from the beat before to this beat, per row
for row, (record, peak) in enumerate(zip(b["record"], b["r_peak"])):
    p = peaks[record]
    k = np.searchsorted(p, peak)
    if k > 0:  # the first beat of a record has no beat before it
        before[row] = (peak - p[k - 1]) / 360
```

The beat before is the one in the windows, not the previous row of the handed-out dataset,
which has no F and Q beats. A study may also build on the windows instead of on single
beats, as long as the split stays by patient and the validation patients stay unseen.

Questions to brainstorm with, before you build anything; the analysis of 2.2.5 answers the
first of them for S, and the rest are yours. Write your answers down; they are your
hypothesis.

- **What would you look at, if you were the cardiologist?** Besides the shape of this beat:
  what comes before it, what after it, and compared with what?
- **"Early" is relative to a heart.** A heart that beats at 50 and one at 110 have another
  normal interval. What would you divide by? Does it use labels? Is it known on a live
  patient from the first beat, or only afterwards?
- **Take the normal beat as the default.** About nine in ten beats are N, so a patient's own
  normal beat and normal interval can be estimated without labels. What is left when you
  subtract that beat from a new beat, and when you divide the interval by the normal one?
  Train a model on what is left and not on the whole signal, as N-BEATS subtracts what a
  block has explained (notebook 2.3). Which classes would you expect to stand out in the
  residual of the shape, and which only in that of the timing?
- **Where does it go in the design?** A number next to the flattened beat in the head? A
  second channel? The intervals of the last few beats as a sequence for a state? A window
  of several beats and a model that labels the one in the middle? Each is a stage or an
  option of `designs.py`; which one your idea needs is yours to say.
- **What does it cost?** A rule that flags every early beat finds S and V and raises false
  alarms on N. Read the recall per class and the confusion matrix next to the macro recall;
  decide what the false alarms are worth.

## The open questions

Every block is a bet about the data. Each question below is one bet a study can test;
the class spreads over them, so that lesson 3 opens on a map of which bets paid:

- **Does V need a wider receptive field than N?** Depth, kernel size and downsample
  change the field; read the recall of V next to the macro recall.
- **Does a state replace depth?** Notebook 2.2 put a GRU after two blocks and lost. With
  a search over depth and width, does the state pay anywhere?
- **Does the fold pay for S?** Notebook 2.1 found the fold no better than 1D on the
  macro recall. Is that also true of S's recall?
- **Does a weighted loss raise S's recall without breaking N's?** E5 measured one point,
  `class_weight` 1, on one config. Where between 0 and 1, and on which designs?
- **Does the rhythm find S where the shape does not?** An S beat comes too early after
  the beat before it, and one row of this data does not hold that. Every beat carries
  its `record` and its `r_peak` (the sample of its R-peak at 360 Hz), so the interval to
  the previous beat of the same patient can be computed. How you get it into the model
  is your design.
- **Does a calibration per patient help?** "Too early" and "odd-shaped" mean: compared
  with this heart. A recording lasts half an hour, and about nine in ten beats are N, so
  the first minute, without its labels, estimates this patient's normal beat and normal
  interval. What would you compute from that minute, and where in the design does it
  go: as an input, a feature for the head, or a way to scale the beat? Of a validation
  patient, use only beats, never labels.
- **Is S a shape of its own, or mostly *not N and not V*?** The twenty beats per class
  of notebook 2.1 (slide 17) show N and V clearly and S hardly. If S is mostly "unlike a
  normal beat", a design can say so: a frozen similarity to normal beats as a feature
  (of all patients, or of this patient's own beats?), one route that only recognises N,
  or several experts behind a router that picks one per beat. Do the experts then
  specialise by themselves, or does the router send everything to one? Lesson 3 comes
  back to that collapse (goal [3.4](../../pages/leerdoelen/les3.html#g3-4));
  a study here is the first measurement of it.
- **Do two kernels side by side see more than one?** `parallel` puts a kernel of 15
  beside the first; is that a wider field for less depth?

A question of your own is welcome, as long as it is a bet you can state before you run.

## The steps

Run every command in a terminal, from the root of the repository.

1. **Design.** Read the design in atlas:

   ```bash
   atlas-code-nightly view notebooks/2_kernels_state_and_gates/exercises
   ```

2. **Build.** Change the design if your question needs a block it does not have, then check
   that every combination builds:

   ```bash
   uv run notebooks/2_kernels_state_and_gates/exercises/designs.py
   ```

3. **Baseline.** Nothing to do: `make study LESSON=2` measures the linear baseline before it
   searches.
4. **Budget and space.** `STUDY` is your first name and what it studies:
   `"sam-state-after-kernels"`. `IDEA` says what you investigate in your own words; your
   first idea builds on Part 1 (`builds_on=Idea.handed_out(2).name`), a later one on an
   idea of the class, by its name. `EXPECTS` is what you expect, and why, written before
   you run. `SPACE` searches the axes your question needs and fixes the rest; every
   option of the design is an axis, searched or fixed. `MINUTES` is your budget; the
   cell below shows what it buys.
5. **Search:** `make study LESSON=2`. It searches, reruns the winner and prints it against
   the baseline in points.
6. **Read** it in `make optuna-dashboard`: per axis, what it did to the macro recall and
   to `train_seconds`, and the recall of S of your best trials.
7. **Explain:** `make page LESSON=2` builds your page of the lesson from your store; its
   step on the page says what to fill in. The page is a starting point: if you want to see
   your runs another way, build your own, better dashboard with Claude, from
   `scripts/study_page.py` or from scratch, on the same store. Keep every score next to its
   baseline, and no axis that mixes two datasets.
8. **Share:** `make share LESSON=2`.
9. **Build on:** see the class in `make dashboard LESSON=2`. Your next idea builds on one
   of the class.

In [ ]:
MINUTES = 15  # the time you give your study on this machine
AXES = 4  # how many axes your SPACE searches
PART1 = StudyLog.load(name="lesson2-notebook")
SECONDS_PER_TRIAL = PART1.seconds_per_trial(epochs=EPOCHS)


def trials_in(minutes: float) -> int:
    """The trials that fit in `minutes`, once the baseline and the winner's rerun
    (REPEATS runs each) are set aside."""
    return Budget.at(
        minutes=minutes,
        seconds_per_trial=SECONDS_PER_TRIAL,
        measured_on=PART1.name,
        epochs=EPOCHS,
        repeats=REPEATS,
    ).n_trials


minutes = np.arange(1, 61)
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(minutes, [trials_in(m) for m in minutes], color="tab:blue")
ax.scatter([MINUTES], [trials_in(MINUTES)], color="tab:orange", s=80, zorder=3)
ax.annotate(
    f"{MINUTES} minutes: {trials_in(MINUTES)} trials, "
    f"{trials_in(MINUTES) / AXES:.0f} per searched axis",
    (MINUTES, trials_in(MINUTES)),
    xytext=(10, -14),
    textcoords="offset points",
)
ax.set_xlabel("minutes you give the study")
ax.set_ylabel("trials that fit")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(
    f"one trial costs {SECONDS_PER_TRIAL:.1f}s here, at {EPOCHS} epochs "
    "(measured by the search of Part 1)",
    loc="left",
    fontsize=10,
)
plt.show()

## Hand in your runs before the next lesson

```bash
make share LESSON=2
```

It hands in Part 1 (`lesson2-notebook`, `lesson2-notebook-runs`) and every study of
yours (`lesson2-study-*`), each run with the design script it ran. Name the ideas of
classmates you built on in `builds_on`; the class page shows who built on whom.